# Persistent junction capture

Ordinary contacts are transient. A junction exists only when a recipe
explicitly promotes an eligible contact, typically after the geometry
is substantially relaxed. Anchors use material coordinates so they
survive adaptive remeshing.

In [ ]:
# Junction angles are expressed in radians.
import math
import tangle

## Every `JunctionPolicy` field

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `name` | Human-readable capture-policy name. | nonempty string |
| `law_name` | Symbolic downstream junction law. | nonempty string |
| `parameter_set` | Junction-law parameter-table identifier. | integer |
| `maximum_surface_gap` | Largest surface gap eligible for capture. | m |
| `minimum_crossing_angle` | Smallest accepted unsigned crossing angle. | rad, 0 to pi/2 |
| `maximum_crossing_angle` | Largest accepted unsigned crossing angle. | rad, 0 to pi/2 |
| `probability` | Deterministic seeded thinning probability. | 0–1 |
| `seed` | Seed for probabilistic capture. | integer |
| `material_pairs` | Optional allowed material-name pairs. | list of pairs |
| `maximum_per_fiber_pair` | Maximum anchors captured between one fiber pair. | positive count |
| `minimum_anchor_separation` | Required material-coordinate separation between anchors. | m |
| `candidate_capacity` | Device candidate-buffer capacity. | positive count |

In [ ]:
# Read defaults from the compiled extension instead of duplicating
# them in documentation that could become stale.
policy = tangle.JunctionPolicy()
fields = ['name', 'law_name', 'parameter_set', 'maximum_surface_gap', 'minimum_crossing_angle', 'maximum_crossing_angle', 'probability', 'seed', 'material_pairs', 'maximum_per_fiber_pair', 'minimum_anchor_separation', 'candidate_capacity']
{name: getattr(policy, name) for name in fields}

In [ ]:
# The law name and parameter set are downstream labels; capture filters
# decide which current contacts receive persistent material anchors.
policy.name = "cured binder contacts"
policy.law_name = "cohesive bond"
policy.parameter_set = 2
policy.maximum_surface_gap = 0.2e-6
policy.minimum_crossing_angle = math.radians(20)
policy.maximum_crossing_angle = math.pi / 2
# Probability is deterministic for a fixed seed and candidate set.
policy.probability = 0.25
policy.seed = 9
policy.material_pairs = [("large", "small"), ("large", "large")]
policy.maximum_per_fiber_pair = 1
policy.minimum_anchor_separation = 50e-6
policy.candidate_capacity = 100_000

`capture_junctions(policy)` samples once at that recipe point.
`relax_and_capture(iterations, every, policy)` samples repeatedly at an
explicit cadence independent of scheduler batch size. Captured
junctions are topology/export data; current relaxation does not enforce
their mechanics, so late capture is normally the physically appropriate
workflow.

In [ ]:
recipe = tangle.Recipe(tangle.Cell([1e-3, 1e-3, 1e-3]))
# Late capture records bonds after geometry has settled; contacts before
# this explicit operation remain transient.
recipe.relax(maximum_iterations=5_000)
recipe.capture_junctions(policy)
# Alternative repeated capture:
# recipe.relax_and_capture(iterations=2_000, every=250, policy=policy)
recipe.operations()